# Data Pipeline — PDF → chunks/images/tables → Qdrant

Chạy 1 lần từ trên xuống. Xử lý **1 PDF** (đổi `PDF_INDEX` để chọn PDF khác).
Output: `output/` (chunks/, images/, tables/, `chunks_metadata.json`, `section_tree.json`)
+ collection `rag_docs` trong Qdrant (hybrid dense + BM25).

Giữ lại chỉ những gì chunking + embedding thực sự cần: convert (Docling),
trích ảnh/bảng, HybridChunker 512 token, cây section, embed dense + sparse,
upsert id ổn định (uuid5 theo chunk_id) để chạy lại nhiều PDF không bị trùng.
Bỏ: file debug 25 MB, image bytes trong RAM, kiểm tra vision không dùng,
các cell tạo collection trùng lặp và cell chat-model không liên quan.


In [1]:
# ============================================================
# 0. SETUP + CONFIG — một nguồn sự thật duy nhất
# ============================================================
import json
import os
import re
import uuid
from collections import defaultdict
from datetime import datetime
from pathlib import Path

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

PDF_DIR = Path("docs")
DOC_VERSION = "v1"
OUTPUT_DIR = Path("output")
CHUNKS_DIR = OUTPUT_DIR / "chunks"
IMAGES_DIR = OUTPUT_DIR / "images"
TABLES_DIR = OUTPUT_DIR / "tables"

EMBED_MODEL_ID = "RizwanMalik/qwen3-vl-embedding-2b:q4_0-q8_0"
EMBED_MODEL = EMBED_MODEL_ID
MAX_TOKENS = 512

QDRANT_URL = "http://localhost:6333"
COLLECTION = "rag_docs"
OLLAMA_URL = "http://localhost:11434"
TEXT_BATCH = 16

VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "sparse"

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

PDF_FILES = sorted(p for p in PDF_DIR.rglob("*.pdf") if p.is_file())
if not PDF_FILES:
    raise FileNotFoundError(f"Không tìm thấy file PDF nào trong: {PDF_DIR.resolve()}")

for i, pdf_path in enumerate(PDF_FILES, start=1):
    print(f"[{i}/{len(PDF_FILES)}] {pdf_path.name}")

PDF_INDEX = 0  # đổi index để chọn PDF khác trong docs/
PDF_PATH = PDF_FILES[PDF_INDEX]
DOC_SLUG = re.sub(r"[^A-Za-z0-9_-]+", "", PDF_PATH.stem.replace(" ", "_"))
SOURCE_FILE = PDF_PATH.name
CREATED_AT = datetime.now().astimezone().isoformat(timespec="seconds")
print(f"SELECTED: {PDF_PATH.name} -> {DOC_SLUG}")


def make_doc_slug(pdf_path: Path) -> str:
    return re.sub(r"[^A-Za-z0-9_-]+", "", pdf_path.stem.replace(" ", "_"))


def make_chunk_id(doc_slug: str, version: str, index: int) -> str:
    return f"{doc_slug}-{version}-{index:04d}"


def point_id_for(chunk_id: str) -> str:
    # Point ID ổn định theo chunk_id (uuid5) -> chạy lại PDF khác không bị trùng id.
    return str(uuid.uuid5(uuid.NAMESPACE_URL, chunk_id))


def make_section_path(headings) -> str:
    return " > ".join(headings) if headings else ""


def make_node_id(source_file: str, section_path: str) -> str:
    import hashlib

    digest = hashlib.md5(f"{source_file}::{section_path}".encode("utf-8")).hexdigest()[:12]
    return f"parent_{digest}"


def normalize_headings(headings) -> list:
    result = []
    for heading in headings or []:
        if heading is None:
            continue
        value = heading.strip() if isinstance(heading, str) else str(heading).strip()
        if value:
            result.append(value)
    return result


def heading_info(source_file: str, headings: list):
    # (section_path, section_id, parent_id, parent_title, ancestor_ids, depth)
    if not headings:
        root = make_node_id(source_file, "__ROOT__")
        return "__ROOT__", root, None, "__ROOT__", [], 0
    section_path = make_section_path(headings)
    section_id = make_node_id(source_file, section_path)
    if len(headings) == 1:
        parent_id, parent_title = make_node_id(source_file, "__ROOT__"), "__ROOT__"
    else:
        parent_path = make_section_path(headings[:-1])
        parent_id = make_node_id(source_file, parent_path)
        parent_title = headings[-2]
    ancestor_ids = [make_node_id(source_file, make_section_path(headings[:k])) for k in range(1, len(headings))]
    return section_path, section_id, parent_id, parent_title, ancestor_ids, len(headings)


def get_chunk_pages(chunk) -> list:
    pages = set()
    for item in chunk.meta.doc_items:
        for prov in getattr(item, "prov", []) or []:
            page_no = getattr(prov, "page_no", None)
            if page_no is not None:
                pages.add(page_no)
    return sorted(pages)


def get_chunk_page_boxes(chunk) -> dict:
    page_boxes = defaultdict(list)
    for item in chunk.meta.doc_items:
        for prov in getattr(item, "prov", []) or []:
            page_no = getattr(prov, "page_no", None)
            if page_no is None:
                continue
            bbox = getattr(prov, "bbox", None)
            if bbox is not None:
                page_boxes[page_no].append(bbox)
    return {
        page_no: {
            "left": min(b.l for b in boxes),
            "right": max(b.r for b in boxes),
            "bottom": min(b.b for b in boxes),
            "top": max(b.t for b in boxes),
        }
        for page_no, boxes in page_boxes.items()
    }


def vertical_distance(picture_bbox, chunk_box) -> float:
    pt, pb = picture_bbox.t, picture_bbox.b
    ct, cb = chunk_box["top"], chunk_box["bottom"]
    if pb <= ct and pt >= cb:
        return 0.0
    if pb > ct:
        return pb - ct
    return cb - pt


[1/3] Z7Z6UMNSG_NSGTH(En)07.pdf
[2/3] ZfcUMEUR_(En)06.pdf
[3/3] ZfUMEUR_(En)05.pdf
SELECTED: Z7Z6UMNSG_NSGTH(En)07.pdf -> Z7Z6UMNSG_NSGTHEn07


In [2]:
# ============================================================
# 1. CONVERT PDF bằng Docling
# ============================================================
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.do_table_structure = True
pipeline_options.generate_picture_images = True
pipeline_options.generate_page_images = False
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

print(f"Processing: {PDF_PATH}")
result = converter.convert(source=PDF_PATH)
doc = result.document
print("PDF parsed successfully.")


2026-10-21 09:36:36,437 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


Processing: docs\Z7Z6UMNSG_NSGTH(En)07.pdf


2026-10-21 09:36:40,581 - INFO - Going to convert document batch...
2026-10-21 09:36:40,696 - INFO - Initializing pipeline for StandardPdfPipeline with options hash cdf150a01134d559192924d442aefd97
2026-10-21 09:36:58,016 - WARNING - From d:\anaconda3\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.

2026-10-21 09:37:00,194 - INFO - Loading plugin 'docling_defaults'
2026-10-21 09:37:00,223 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-10-21 09:37:00,224 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-10-21 09:37:00,274 - INFO - Loading plugin 'docling_defaults'
2026-10-21 09:37:00,328 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-10-21 09:37:00,329 - INFO - Registered ocr

PDF parsed successfully.


In [3]:
# ============================================================
# 2. TRÍCH XUẤT HÌNH + BẢNG -> output/images, output/tables
# ============================================================
from docling_core.types.doc import PictureItem, TableItem

print("Extracting images...")
image_metadata = []
for element, _level in doc.iterate_items():
    if not isinstance(element, PictureItem):
        continue
    n = len(image_metadata) + 1
    image_id = f"picture_{n:04d}"
    image_name = f"{DOC_SLUG}_img_{n:04d}_{DOC_VERSION}.png"
    image_path = IMAGES_DIR / image_name

    doc_item_ref = None
    page_no = None
    try:
        image = element.get_image(doc)
        image.save(image_path, "PNG")
        doc_item_ref = str(element.self_ref)
        if element.prov:
            page_no = element.prov[0].page_no
        image_metadata.append({
            "image_id": image_id,
            "file": image_path.relative_to(OUTPUT_DIR).as_posix(),
            "source_file": SOURCE_FILE,
            "created_at": CREATED_AT,
            "doc_item_ref": doc_item_ref,
            "page_no": page_no,
            "width": image.width,
            "height": image.height,
        })
    except Exception as e:
        print(f"Failed image {n}: {e}")

(OUTPUT_DIR / "images_metadata.json").write_text(
    json.dumps(image_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Extracted {len(image_metadata)} images -> {IMAGES_DIR}")

print("Extracting tables...")
table_metadata = []
for element, _level in doc.iterate_items():
    if not isinstance(element, TableItem):
        continue
    n = len(table_metadata) + 1
    table_id = f"table_{n:04d}"
    table_name = f"{DOC_SLUG}_tbl_{n:04d}_{DOC_VERSION}"
    csv_path = TABLES_DIR / f"{table_name}.csv"
    html_path = TABLES_DIR / f"{table_name}.html"

    try:
        doc_item_ref = str(element.self_ref)
    except Exception:
        doc_item_ref = None
    page_no = None
    try:
        if element.prov:
            page_no = element.prov[0].page_no
    except Exception:
        pass
    df = None
    try:
        df = element.export_to_dataframe(doc=doc)
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    except Exception as e:
        print(f"Table {table_id} CSV failed: {e}")
    try:
        html_path.write_text(element.export_to_html(doc=doc), encoding="utf-8")
    except Exception as e:
        print(f"Table {table_id} HTML failed: {e}")

    table_metadata.append({
        "table_id": table_id,
        "source_file": SOURCE_FILE,
        "created_at": CREATED_AT,
        "doc_item_ref": doc_item_ref,
        "page_no": page_no,
        "csv": csv_path.relative_to(OUTPUT_DIR).as_posix() if csv_path.exists() else None,
        "html": html_path.relative_to(OUTPUT_DIR).as_posix() if html_path.exists() else None,
        "rows": len(df) if df is not None else None,
        "columns": len(df.columns) if df is not None else None,
        "column_names": [str(c) for c in df.columns] if df is not None else [],
    })
(OUTPUT_DIR / "tables_metadata.json").write_text(
    json.dumps(table_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Extracted {len(table_metadata)} tables -> {TABLES_DIR}")


Extracting images...
Extracted 403 images -> output\images
Extracting tables...
Extracted 107 tables -> output\tables


In [12]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import ollama

from docling.chunking import HybridChunker


# ============================================================
# 2. CONFIGURATION
# ============================================================

EMBED_MODEL_ID = "RizwanMalik/qwen3-vl-embedding-2b:q4_0-q8_0"

MAX_TOKENS = 512


# ============================================================
# 3. CHUNK DOCUMENT
# ============================================================

print("Initializing HybridChunker...")

# Sử dụng tokenizer mặc định của Docling
chunker = HybridChunker(
    merge_peers=True
)

print("Chunking document...")

chunks = list(chunker.chunk(dl_doc=doc))

print(f"Total chunks: {len(chunks)}")


# ============================================================
# 4. EXTRACT TEXT FROM CHUNKS
# ============================================================

texts = [chunk.text for chunk in chunks]

print(f"Total texts: {len(texts)}")


# ============================================================
# 5. GENERATE EMBEDDINGS USING LOCAL OLLAMA
# ============================================================

print("Generating embeddings with Ollama...")

response = ollama.embed(
    model=EMBED_MODEL_ID,
    input=texts
)

embeddings = response["embeddings"]

print(f"Total embeddings: {len(embeddings)}")


# ============================================================
# 6. VERIFY RESULTS
# ============================================================

if len(chunks) != len(embeddings):
    raise ValueError("Number of chunks does not match number of embeddings.")

print("\nEmbedding completed successfully!")
print(f"Number of chunks: {len(chunks)}")
print(f"Embedding dimensions: {len(embeddings[0])}")

# Inspect the first chunk
print("\nFirst chunk:")
print(chunks[0].text[:500])

print("\nFirst embedding:")
print(embeddings[0][:10])

Initializing HybridChunker...


Token indices sequence length is longer than the specified maximum sequence length for this model (2024 > 512). Running this sequence through the model will result in indexing errors


Chunking document...
Total chunks: 668
Total texts: 668
Generating embeddings with Ollama...


2026-10-21 20:47:41,697 - INFO - HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


Total embeddings: 668

Embedding completed successfully!
Number of chunks: 668
Embedding dimensions: 2048

First chunk:
DIGITAL CAMERA
User's Manual
This manual is for camera firmware versions 3.00 and later. The latest version of the camera firmware is available for download from the Nikon Download Center.
- Read this manual thoroughly before using the camera.
- To ensure proper use of the camera, be sure to read 'For Your Safety' (page ix).
- After reading this manual, keep it in a readily accessible place for future reference.
Scan for detailed operating instructions.
nikon online manual Z 7
Z 6

First embedding:
[-0.039726425, -0.008335508, 0.012000322, 0.02150678, -0.031792473, 0.01596363, -0.011116004, 0.030573742, 0.019875994, -0.0038692486]


In [13]:
print(type(chunks), len(chunks))

<class 'list'> 668


In [14]:
# ============================================================
# 4. BUILD CHUNKS + SECTION TREE
# ============================================================
import json
from collections import defaultdict
from docling_core.types.doc import PictureItem

image_map = {m["doc_item_ref"]: m for m in image_metadata if m.get("doc_item_ref")}
table_map = {m["doc_item_ref"]: m for m in table_metadata if m.get("doc_item_ref")}

chunk_boxes_list = [get_chunk_page_boxes(c) for c in chunks]
pictures_by_page = defaultdict(list)

for element, _level in doc.iterate_items():
    if not isinstance(element, PictureItem):
        continue
    for prov in element.prov or []:
        page_no = getattr(prov, "page_no", None)
        bbox = getattr(prov, "bbox", None)
        if page_no is not None and bbox is not None:
            pictures_by_page[page_no].append({
                "picture_ref": str(element.self_ref),
                "bbox": bbox,
            })

picture_to_chunk = {}
for page, pics in pictures_by_page.items():
    for pic in pics:
        best, best_d = None, float("inf")
        for idx, boxes in enumerate(chunk_boxes_list):
            if page not in boxes:
                continue
            d = vertical_distance(pic["bbox"], boxes[page])
            if d < best_d:
                best_d, best = d, idx
        if best is not None:
            ref = pic["picture_ref"]
            prev = picture_to_chunk.get(ref)
            if prev is None or best_d < prev["distance"]:
                picture_to_chunk[ref] = {"chunk_index": best, "distance": best_d}

bbox_refs_by_chunk = defaultdict(list)
for ref, mapping in picture_to_chunk.items():
    bbox_refs_by_chunk[mapping["chunk_index"]].append(ref)

raw_chunks, last_headings = [], []

for index, chunk in enumerate(chunks):
    chunk_id = make_chunk_id(DOC_SLUG, DOC_VERSION, index)
    text = chunker.contextualize(chunk=chunk)
    num_tokens = chunker.tokenizer.count_tokens(text)

    raw_headings = normalize_headings(getattr(chunk.meta, "headings", None))
    current_headings = raw_headings if raw_headings else last_headings
    if raw_headings:
        last_headings = raw_headings

    section_path, section_id, parent_id, parent_title, ancestor_ids, depth = heading_info(
        SOURCE_FILE, current_headings
    )
    section_title = current_headings[-1] if current_headings else "__ROOT__"

    raw_chunks.append({
        "index": index, "chunk": chunk, "chunk_id": chunk_id,
        "text": text, "num_tokens": num_tokens,
        "headings": current_headings, "section_path": section_path,
        "section_id": section_id, "section_title": section_title,
        "parent_id": parent_id, "parent_title": parent_title,
        "ancestor_ids": ancestor_ids,
        "ancestor_titles": current_headings[:-1], "depth": depth,
        "doc_items": [
            {"ref": str(it.self_ref), "label": str(it.label)}
            for it in chunk.meta.doc_items
        ],
        "page_nos": get_chunk_pages(chunk),
    })

parent_groups, section_nodes = defaultdict(list), {}

for item in raw_chunks:
    sid = item["section_id"]
    parent_groups[sid].append(item["index"])
    if sid not in section_nodes:
        section_nodes[sid] = {
            "node_id": sid, "title": item["section_title"],
            "level": item["depth"], "parent_id": item["parent_id"],
            "section_path": item["section_path"],
            "ancestor_ids": item["ancestor_ids"], "chunk_ids": [],
        }
    section_nodes[sid]["chunk_ids"].append(item["chunk_id"])

for sid, node in section_nodes.items():
    pid = node["parent_id"]
    if pid and pid in section_nodes:
        section_nodes[pid].setdefault("child_node_ids", []).append(sid)

for node in section_nodes.values():
    node.setdefault("child_node_ids", [])

doc_root = make_node_id(SOURCE_FILE, "__ROOT__")
section_nodes[doc_root] = {
    "node_id": doc_root, "title": SOURCE_FILE, "level": 0,
    "parent_id": None, "section_path": "__ROOT__",
    "ancestor_ids": [], "chunk_ids": [],
    "child_node_ids": [
        sid for sid, n in section_nodes.items()
        if n.get("parent_id") == doc_root
    ],
}

chunk_metadata = []

for item in raw_chunks:
    index, chunk_id = item["index"], item["chunk_id"]
    exact_images, exact_tables = [], []
    exact_image_refs, exact_table_refs = set(), set()

    for doc_item in item["chunk"].meta.doc_items:
        ref = str(doc_item.self_ref)

        if ref in image_map and ref not in exact_image_refs:
            m = image_map[ref]
            exact_images.append({
                "image_id": m["image_id"], "file": m["file"],
                "page_no": m["page_no"], "doc_item_ref": ref,
                "match": "exact",
            })
            exact_image_refs.add(ref)

        if ref in table_map and ref not in exact_table_refs:
            m = table_map[ref]
            exact_tables.append({
                "table_id": m["table_id"], "csv": m["csv"],
                "html": m["html"], "page_no": m["page_no"],
                "doc_item_ref": ref,
            })
            exact_table_refs.add(ref)

    bbox_images = [
        {
            "image_id": image_map[ref]["image_id"],
            "file": image_map[ref]["file"],
            "page_no": image_map[ref]["page_no"],
            "doc_item_ref": ref,
            "match": "bbox",
        }
        for ref in bbox_refs_by_chunk.get(index, [])
        if ref not in exact_image_refs and ref in image_map
    ]

    images, tables = exact_images + bbox_images, exact_tables
    section_id = item["section_id"]

    chunk_metadata.append({
        "chunk_id": chunk_id, "chunk_index": index,
        "source_file": SOURCE_FILE, "section_id": section_id,
        "section_title": item["section_title"],
        "parent_id": item["parent_id"], "parent_title": item["parent_title"],
        "section_path": item["section_path"],
        "ancestor_ids": item["ancestor_ids"],
        "page_nos": item["page_nos"], "images": images, "tables": tables,
    })

    front_matter = {
        "chunk_id": chunk_id, "chunk_index": index,
        "source_file": SOURCE_FILE, "section_id": section_id,
        "section_title": item["section_title"],
        "parent_id": item["parent_id"] or "",
        "parent_title": item["parent_title"],
        "section_path": item["section_path"], "depth": item["depth"],
        "created_at": CREATED_AT, "num_tokens": item["num_tokens"],
    }

    yaml_lines = ["---"]
    for key, value in front_matter.items():
        if isinstance(value, str):
            value = value.replace("\\", "\\\\").replace('"', '\\"')
            yaml_lines.append(f'{key}: "{value}"')
        else:
            yaml_lines.append(f"{key}: {value}")
    yaml_lines.append("---")

    (CHUNKS_DIR / f"{chunk_id}.md").write_text(
        "\n".join(yaml_lines) + "\n\n" + item["text"] + "\n",
        encoding="utf-8",
    )

    print(f"[{index + 1}/{len(raw_chunks)}] {chunk_id} "
          f"({item['num_tokens']} tokens) images={len(images)} tables={len(tables)}")

(OUTPUT_DIR / "chunks_metadata.json").write_text(
    json.dumps(chunk_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "section_tree.json").write_text(
    json.dumps({
        "root_id": doc_root, "source_file": SOURCE_FILE, "nodes": section_nodes
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Saved {len(chunk_metadata)} chunks -> {OUTPUT_DIR / 'chunks_metadata.json'}")
print(f"Unique sections: {len(parent_groups)} | section nodes: {len(section_nodes)}")

del chunks, chunk_boxes_list

[1/668] Z7Z6UMNSG_NSGTHEn07-v1-0000 (103 tokens) images=5 tables=0
[2/668] Z7Z6UMNSG_NSGTHEn07-v1-0001 (246 tokens) images=1 tables=0
[3/668] Z7Z6UMNSG_NSGTHEn07-v1-0002 (218 tokens) images=0 tables=0
[4/668] Z7Z6UMNSG_NSGTHEn07-v1-0003 (148 tokens) images=0 tables=0
[5/668] Z7Z6UMNSG_NSGTHEn07-v1-0004 (152 tokens) images=3 tables=0
[6/668] Z7Z6UMNSG_NSGTHEn07-v1-0005 (47 tokens) images=1 tables=0
[7/668] Z7Z6UMNSG_NSGTHEn07-v1-0006 (54 tokens) images=1 tables=0
[8/668] Z7Z6UMNSG_NSGTHEn07-v1-0007 (85 tokens) images=4 tables=0
[9/668] Z7Z6UMNSG_NSGTHEn07-v1-0008 (46 tokens) images=4 tables=0
[10/668] Z7Z6UMNSG_NSGTHEn07-v1-0009 (172 tokens) images=4 tables=0
[11/668] Z7Z6UMNSG_NSGTHEn07-v1-0010 (77 tokens) images=1 tables=0
[12/668] Z7Z6UMNSG_NSGTHEn07-v1-0011 (24 tokens) images=0 tables=0
[13/668] Z7Z6UMNSG_NSGTHEn07-v1-0012 (181 tokens) images=3 tables=0
[14/668] Z7Z6UMNSG_NSGTHEn07-v1-0013 (14 tokens) images=0 tables=0
[15/668] Z7Z6UMNSG_NSGTHEn07-v1-0014 (28 tokens) images=0 tables

In [15]:
# ============================================================
# 5. QDRANT + embedding clients; tạo collection (idempotent)
# ============================================================
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import FastEmbedSparse

client = QdrantClient(url=QDRANT_URL)
embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

if not client.collection_exists(COLLECTION):
    dim = len(embeddings.embed_query("dimension test"))
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config={VECTOR_NAME: VectorParams(size=dim, distance=Distance.COSINE)},
        sparse_vectors_config={SPARSE_VECTOR_NAME: SparseVectorParams()},
    )
    print(f"Created collection '{COLLECTION}' dense={dim}+sparse=BM25")
else:
    info = client.get_collection(COLLECTION)
    print("Collection already exists")

for key, schema in [
    ("metadata.section_id", "keyword"),
    ("metadata.ancestor_ids", "keyword"),
    ("metadata.page_nos", "integer"),
]:
    try:
        client.create_payload_index(collection_name=COLLECTION, field_name=key, field_schema=schema)
        print(f"index ok: {key}")
    except Exception as e:
        print(f"index skip {key}: {e}")


2026-10-21 20:50:07,240 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-10-21 20:50:08,321 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs/exists "HTTP/1.1 200 OK"
2026-10-21 20:50:08,359 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"
2026-10-21 20:50:08,420 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs/index?wait=true "HTTP/1.1 200 OK"
2026-10-21 20:50:08,460 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs/index?wait=true "HTTP/1.1 200 OK"
2026-10-21 20:50:08,482 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs/index?wait=true "HTTP/1.1 200 OK"


Collection already exists
index ok: metadata.section_id
index ok: metadata.ancestor_ids
index ok: metadata.page_nos


In [16]:
# ============================================================
# 6. INGEST: embed dense + BM25 rồi upsert vào Qdrant
# ============================================================
from pathlib import Path
from qdrant_client.models import PointStruct, SparseVector

OUTPUT_DIR = Path(OUTPUT_DIR)
CHUNKS_DIR = Path(CHUNKS_DIR)

chunks_meta = json.loads((OUTPUT_DIR / "chunks_metadata.json").read_text(encoding="utf-8"))
tables_meta = json.loads((OUTPUT_DIR / "tables_metadata.json").read_text(encoding="utf-8"))

table_csv = {}
for table in tables_meta:
    if table.get("csv"):
        path = OUTPUT_DIR / table["csv"]
        if path.exists():
            table_csv[table["table_id"]] = path.read_text(encoding="utf-8-sig")


def chunk_body(chunk_id: str) -> str:
    raw = (CHUNKS_DIR / f"{chunk_id}.md").read_text(encoding="utf-8")
    parts = raw.split("---\n")
    return "---\n".join(parts[2:]).strip() if len(parts) >= 3 else raw.strip()


def get_existing_ids() -> set:
    existing = set()
    offset = None
    while True:
        points, offset = client.scroll(
            collection_name=COLLECTION,
            limit=512,
            offset=offset,
            with_vectors=False,
            with_payload=False,
        )
        existing.update(str(p.id) for p in points)
        if offset is None:
            break
    return existing


existing = get_existing_ids()
todo = [m for m in chunks_meta if point_id_for(m["chunk_id"]) not in existing]
print(f"Existing points: {len(existing)} | Chunks to embed: {len(todo)}/{len(chunks_meta)}")

if todo:
    bodies = []
    inputs = []
    for m in todo:
        body = chunk_body(m["chunk_id"])
        bodies.append(body)
        extra = ""
        for table in m.get("tables", []):
            table_id = table["table_id"]
            if table_id in table_csv:
                extra += f"\n\n[TABLE {table_id} page {table.get('page_no')}]\n{table_csv[table_id]}"
        inputs.append(body + extra)

    total_batches = (len(inputs) + TEXT_BATCH - 1) // TEXT_BATCH
    for start in range(0, len(inputs), TEXT_BATCH):
        batch = inputs[start:start + TEXT_BATCH]
        batch_meta = todo[start:start + TEXT_BATCH]
        dense = embeddings.embed_documents(batch)
        sparse = sparse_embeddings.embed_documents(batch)
        points = []
        for m, body, dv, sv in zip(batch_meta, batch, dense, sparse):
            payload = {
                "page_content": body,
                "metadata": m,
            }
            points.append(
                PointStruct(
                    id=point_id_for(m["chunk_id"]),
                    vector={
                        VECTOR_NAME: dv,
                        SPARSE_VECTOR_NAME: SparseVector(indices=list(sv.indices), values=list(sv.values)),
                    },
                    payload=payload,
                )
            )
        client.upsert(collection_name=COLLECTION, points=points)
        for m in batch_meta:
            existing.add(point_id_for(m["chunk_id"]))
        print(f"  upserted batch {start // TEXT_BATCH + 1}/{total_batches}")

info = client.get_collection(COLLECTION)
print("points:", info.points_count)
assert info.points_count >= len(chunks_meta), "Ingest bỏ sót điểm — kiểm tra lại id đã tồn tại"
print("Ingest done.")


2026-10-21 20:50:13,527 - INFO - HTTP Request: POST http://localhost:6333/collections/rag_docs/points/scroll "HTTP/1.1 200 OK"


Existing points: 0 | Chunks to embed: 668/668


2026-10-21 20:50:24,418 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-10-21 20:50:24,553 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs/points?wait=true "HTTP/1.1 200 OK"


  upserted batch 1/42


2026-10-21 20:50:43,906 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-10-21 20:50:43,992 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs/points?wait=true "HTTP/1.1 200 OK"


  upserted batch 2/42


... (76 outputs left out)

2026-10-21 21:04:43,139 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-10-21 21:04:43,194 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs/points?wait=true "HTTP/1.1 200 OK"


  upserted batch 41/42


2026-10-21 21:04:57,510 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-10-21 21:04:57,571 - INFO - HTTP Request: PUT http://localhost:6333/collections/rag_docs/points?wait=true "HTTP/1.1 200 OK"
2026-10-21 21:04:57,598 - INFO - HTTP Request: GET http://localhost:6333/collections/rag_docs "HTTP/1.1 200 OK"


  upserted batch 42/42
points: 668
Ingest done.
